In [1]:
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory


import pandas as pd
import numpy as np
from src.data import Data
import src.metrics as metrics

from rank_bm25 import BM25Okapi
import string

d:\venv\mm-prod-search-and-reco-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
d1 = Data()
d_train = d1.get_complete_data('train')
d_train

Loading ESIC DATA...
ESIC Downloaded succesfully!
Loading SQID DATA...
SQID Downloaded succesfully!
Loading Amazon Review DATA...
Amazon Review Data Downloaded succesfully!


,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split_x,__index_level_0___x,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale_y,split_y,__index_level_0___y,image_url
0,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,16,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch,us,train,1515633,https://m.media-amazon.com/images/I/81r5MLo6nr...
1,149,# 10 self-seal envelopes without window,5,B07RP3F2LW,us,E,1,1,train,149,"BAZIC Security Self Seal Envelope 4 1/8"" x 9 1...",<p><strong>BACK TO BAZIC</strong></p> <p>Our g...,"&#11088; #10 SECURITY ENVELOPES. These 4 1/8"" ...",BAZIC Products,#10 (30-count),us,train,1387952,https://m.media-amazon.com/images/I/71UzvOXH2K...
2,152,# 10 self-seal envelopes without window,5,B07GJCSF8Z,us,E,1,1,train,152,ValBox 500 Count #10 Security Self-Seal Envelo...,"""About ValBox. </br> </br> Engaged in envelope...",#10 Security Tinted Self Seal Envelopes 500 Co...,ValBox,White-500 Count,us,train,1387947,https://m.media-amazon.com/images/I/71fMUzp0Tw...
3,153,# 10 self-seal envelopes without window,5,B07GJKFC5X,us,E,1,1,train,153,"ValBox #10 Security Envelopes Self Seal, No. 1...","""About ValBox. </br> </br> Engaged in envelope...",#10 Security Tinted Self Seal Envelopes 200 Co...,ValBox,White-200 Count,us,train,1387948,https://m.media-amazon.com/images/I/71TSOB6rDA...
4,155,# 10 self-seal envelopes without window,5,B07NZ4T2SL,us,I,1,1,train,155,ValBox 200 Count #8 Double Window Envelopes 3 ...,"""About ValBox. </br> </br> Engaged in envelope...",Security Check Envelopes. Fits QuickBooks Chec...,ValBox,200 Count,us,train,1387949,https://m.media-amazon.com/images/I/71-UFz+k2p...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39947,1565020,phycocyanin,79682,B098MGLB2C,us,E,0,1,train,1565020,Bwana Butter Blue Spirulina Powder Large 3 oz....,<strong>Blue Spirulina</strong> is the common ...,&#128153; 3 full ounces of blue spirulina powd...,Bwana Butter,Blue,us,test,1377590,https://m.media-amazon.com/images/I/81IDSnfFm1...
39948,1565039,phycocyanin,79682,B06XGW8Z1F,us,I,0,1,train,1565039,BRI Nutrition Klamath Blue Green Algae - More ...,"★★★★★With more protein, B12 and chlorophyll th...",&#x2605;WHY US: One of the biggest complaints ...,BRI Nutrition,1-pack,us,test,1377565,https://m.media-amazon.com/images/I/91Uf7Q+Kzk...
39949,1565292,phytoceramides without wheat,79696,B07SRK1F3Y,us,I,1,1,train,1565292,HumanN SuperBeets Heart Chews | Grape Seed Ext...,SuperBeets Heart Chews puts two of nature’s mo...,Grape seed extract: earned its place in SuperB...,HumanN,Pomegranate-berry,us,test,1726906,https://m.media-amazon.com/images/I/71u2SS4iFA...
39950,1565387,pi hat,79699,B07WD7HXSQ,us,E,0,1,train,1565387,LoveRPi Power-Over-Ethernet (PoE) HAT for Rasp...,<ul> <li>Input: IEEE 802.3af Compliant PoE</li...,ULTIMATE POWER OVER ETHERNET HAT FOR RASPBERRY...,LoveRPi,Non-Isolated,us,train,890244,https://m.media-amazon.com/images/I/81yMa74O32...


In [3]:

cols = [
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color"
]

for col in cols:
    d_train[col]  = d_train[col].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

In [4]:
d_train["product_description"].iloc[0]

['babout',
 'the',
 'rampro',
 'all',
 'purpose',
 'utility',
 '10',
 'air',
 'tires',
 'wheels',
 'pack',
 'of',
 '2bp',
 'bthe',
 'problembbr',
 'your',
 'handtrucks',
 'tires',
 'are',
 'flat',
 'but',
 'why',
 'should',
 'you',
 'spend',
 'so',
 'much',
 'money',
 'to',
 'buy',
 'a',
 'compete',
 'new',
 'handtruckp',
 'bour',
 'solutionbbr',
 'the',
 'rampro',
 'all',
 'purpose',
 '10',
 'air',
 'tires',
 'wheels',
 'that',
 'will',
 'replace',
 'and',
 'fit',
 'your',
 'hand',
 'truck',
 'old',
 'tires',
 'properly',
 'with',
 'very',
 'easy',
 'installations',
 'and',
 'will',
 'be',
 'even',
 'better',
 'and',
 'more',
 'comfortable',
 'than',
 'the',
 'originalsp',
 'bdouble',
 'sealed',
 'bearingsbbr',
 'the',
 'double',
 'bearings',
 'will',
 'help',
 'to',
 'balance',
 'your',
 'loud',
 'evenly',
 'thereby',
 'make',
 'you',
 'push',
 'lass',
 'and',
 'easierp',
 'bsporty',
 'raised',
 'gripsbbr',
 'the',
 'sporty',
 'look',
 'raised',
 'grips',
 'will',
 'help',
 'you',
 '

In [5]:
field_weights = {
    "product_title": 1.0,
    "product_brand": 1.5,
    "product_bullet_point": 2.0,
    "product_description": 2.0,
    "product_color": 1.0
}

bm25 = {}

for col in cols:
    bm25[col] = BM25Okapi(d_train["product_description"].to_list()) 

In [6]:
def get_BM25_score(query, k):
    query_tokens = query.lower().split()

    scores = {}
    for col in cols:
        scores[col] = bm25[col].get_scores(query_tokens)
        
    for col in cols:
        scores[col] = bm25[col].get_scores(query_tokens)

    final_score = sum(
        field_weights[col] * scores[col]
        for col in cols
    )

    top_k_indx = np.argsort(final_score)[::-1][:k]

    return d_train.iloc[top_k_indx]



In [10]:
K = 5
print(d_train["query"].iloc[1])

query = "Shirt" #d_train["query"].iloc[1]
actual_products = d_train[d_train["query"] == query]
retrived_prods = get_BM25_score(d_train["query"].iloc[1], K)

# 10 self-seal envelopes without window


In [11]:
retrived_prods

,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split_x,__index_level_0___x,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale_y,split_y,__index_level_0___y,image_url
56,713,#10 window envelopes without plastic,21,B015773LQK,us,E,1,1,train,713,"[50010, single, left, window, self, seal, secu...","[10, single, window, security, envelopes, grea...","[10, single, window, security, envelopes, forg...",[aimoh],[white],us,test,1387935,https://m.media-amazon.com/images/I/81dB4TWWYY...
2171,35644,10 open window envelopes without plastic window,1273,B015773LQK,us,E,1,1,train,35644,"[50010, single, left, window, self, seal, secu...","[10, single, window, security, envelopes, grea...","[10, single, window, security, envelopes, forg...",[aimoh],[white],us,test,1387935,https://m.media-amazon.com/images/I/81dB4TWWYY...
2168,35634,10 open window envelopes without plastic window,1273,B08V352RY5,us,S,1,1,train,35634,"[500, 10, double, window, self, seal, security...","[500, 10, double, window, self, seal, security...","[double, window, security, envelopes, tired, o...",[aimoh],[white],us,test,1387961,https://m.media-amazon.com/images/I/913bql5M5t...
52,701,#10 window envelopes without plastic,21,B08V352RY5,us,E,1,1,train,701,"[500, 10, double, window, self, seal, security...","[500, 10, double, window, self, seal, security...","[double, window, security, envelopes, tired, o...",[aimoh],[white],us,test,1387961,https://m.media-amazon.com/images/I/913bql5M5t...
28,647,#10 window envelopes not self seal,20,B079H6L3CK,us,E,1,1,train,647,"[self, seal, double, window, security, envelop...","[pbsave, time, and, money, with, the, gsm, off...","[designed, for, business, invoices, statements...","[gsm, brands]",[500],us,test,167299,https://m.media-amazon.com/images/I/41UTHyQXLI...


In [12]:
print(f"Recall@{K}: {metrics.Recall_K(K, retrived_prods, actual_products)}")
print(f"Precision@{K}: {metrics.Precision_K(K, retrived_prods, actual_products)}")
print(f"NDCG@{K}: {metrics.NDCG_K(K, retrived_prods, actual_products)}")
print(f"DCG@{K}: {metrics.DCG_K(K, retrived_prods, actual_products)}")
print(f"Latency: {metrics.query_latency(get_BM25_score, d_train["query"].iloc[1], K)}")

d:\Machine_Learning\Multi-Modal-Product-Search-and-Recommendation-System\src\metrics.py:11: RuntimeWarning: invalid value encountered in scalar divide
  return num_correct_retrieval / total_relevant_prods


Recall@5: nan
Precision@5: 0.0
NDCG@5: 0
DCG@5: 8.345377356638178
Latency: (      example_id                                            query  query_id  \
56           713             #10 window envelopes without plastic        21   
2171       35644  10 open window envelopes without plastic window      1273   
2168       35634  10 open window envelopes without plastic window      1273   
52           701             #10 window envelopes without plastic        21   
28           647               #10 window envelopes not self seal        20   

      product_id product_locale_x esci_label  small_version  large_version  \
56    B015773LQK               us          E              1              1   
2171  B015773LQK               us          E              1              1   
2168  B08V352RY5               us          S              1              1   
52    B08V352RY5               us          E              1              1   
28    B079H6L3CK               us          E              1